# Task 5: Neural Style Transfer (NST) Studio

This notebook demonstrates how to apply the artistic style of one image (e.g., a famous painting) to the content of another image (e.g., a photograph) using **Neural Style Transfer** in PyTorch.

It loads a pre-trained **VGG-19** model, extracts intermediate features, runs an optimization loop to blend the content structure and style textures, and plots the resulting images alongside their respective loss curves over time.

In [ ]:
# 1. Import dependencies and check device acceleration
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

: 

In [ ]:
# 2. Image loader and helper utilities
def load_image(image_path, max_size=256, shape=None):
    """
    Load an image and preprocess it into a PyTorch tensor, applying normalization.
    """
    image = Image.open(image_path).convert('RGB')
    
    if shape:
        size = shape
    else:
        size = max_size
        width, height = image.size
        if max(width, height) > max_size:
            if width > height:
                size = (int(max_size * height / width), max_size)
            else:
                size = (max_size, int(max_size * width / height))
        else:
            size = (height, width)

    in_transform = transforms.Compose([
        transforms.Resize(size),
        transforms.ToTensor(),
        transforms.Normalize((0.485, 0.456, 0.406), 
                             (0.229, 0.224, 0.225))
    ])

    image = in_transform(image).unsqueeze(0)
    return image.to(device)

def im_convert(tensor):
    """
    Convert a normalized PyTorch tensor back to a standard numpy array image.
    """
    image = tensor.cpu().clone().detach().squeeze(0)
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    
    image = image * std + mean
    image = torch.clamp(image, 0, 1)
    return image.numpy().transpose(1, 2, 0)

: 

In [ ]:
# 3. Define VGG-19 feature extractor and Gram matrix calculator
class VGGFeatures(nn.Module):
    def __init__(self):
        super(VGGFeatures, self).__init__()
        try:
            from torchvision.models import VGG19_Weights
            self.vgg = models.vgg19(weights=VGG19_Weights.DEFAULT).features
        except (ImportError, AttributeError):
            self.vgg = models.vgg19(pretrained=True).features
            
        # Freeze weights
        for param in self.vgg.parameters():
            param.requires_grad_(False)
            
        # Target layers map
        self.layers = {
            '0': 'conv1_1',
            '5': 'conv2_1',
            '10': 'conv3_1',
            '15': 'conv4_1',
            '20': 'conv4_2',  # content representation layer
            '21': 'conv5_1'
        }

    def forward(self, x):
        features = {}
        for name, layer in self.vgg._modules.items():
            x = layer(x)
            if name in self.layers:
                features[self.layers[name]] = x
        return features

def gram_matrix(tensor):
    """
    Calculate the Gram matrix of a given tensor to extract style details.
    """
    _, d, h, w = tensor.size()
    tensor = tensor.view(d, h * w)
    gram = torch.mm(tensor, tensor.t())
    return gram.div(d * h * w)

In [ ]:
# 4. Load Content and Style images
content_path = 'images/content.jpg'
style_path = 'images/style.jpg'
output_dir = 'outputs'
os.makedirs(output_dir, exist_ok=True)

content_tensor = load_image(content_path, max_size=256)
style_tensor = load_image(style_path, max_size=256, shape=content_tensor.shape[-2:])

content_np = im_convert(content_tensor)
style_np = im_convert(style_tensor)

# Show inputs side-by-side
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))
ax1.imshow(content_np)
ax1.set_title("Content Target")
ax1.axis('off')
ax2.imshow(style_np)
ax2.set_title("Style Target")
ax2.axis('off')
plt.show()

In [ ]:
# 5. Style Transfer optimization loop
# Hyperparameters matching the comparison plot setup
steps = 120
content_weight = 5000
style_weight = 1e5
tv_weight = 1e-5

# Initialize target image as content image
target_tensor = content_tensor.clone().requires_grad_(True)
model = VGGFeatures().to(device)

content_features = model(content_tensor)
style_features = model(style_tensor)
style_grams = {layer: gram_matrix(style_features[layer]) for layer in style_features}

style_weights = {
    'conv1_1': 1.0,
    'conv2_1': 0.75,
    'conv3_1': 0.2,
    'conv4_1': 0.2,
    'conv5_1': 0.2
}

optimizer = optim.Adam([target_tensor], lr=0.02)

content_losses = []
style_losses = []
total_losses_normalized = []
steps_list = []

print("Starting style transfer optimization...")
for step in range(1, steps + 1):
    optimizer.zero_grad()
    target_features = model(target_tensor)
    
    # Content loss
    c_loss = torch.mean((target_features['conv4_2'] - content_features['conv4_2'])**2)
    
    # Style loss
    s_loss = 0
    for layer in style_weights:
        t_feat = target_features[layer]
        t_gram = gram_matrix(t_feat)
        s_gram = style_grams[layer]
        s_loss += style_weights[layer] * torch.mean((t_gram - s_gram)**2)
        
    # Smoothness loss (TV)
    diff_h = target_tensor[:, :, 1:, :] - target_tensor[:, :, :-1, :]
    diff_w = target_tensor[:, :, :, 1:] - target_tensor[:, :, :, :-1]
    t_loss = torch.mean(diff_h**2) + torch.mean(diff_w**2)
    
    # Combine losses
    total_loss = (content_weight * c_loss) + (style_weight * s_loss) + (tv_weight * t_loss)
    total_loss_norm = total_loss.item() / 1e6
    
    content_losses.append(c_loss.item() * content_weight)
    style_losses.append(s_loss.item())
    total_losses_normalized.append(total_loss_norm)
    steps_list.append(step)
    
    total_loss.backward()
    optimizer.step()

    if step % 20 == 0 or step == steps:
        print(f"Step {step}/{steps} | Content Loss: {content_losses[-1]:.2f} | Style Loss: {style_losses[-1]:.4f} | Total Loss (Norm): {total_loss_norm:.4f}")

output_np = im_convert(target_tensor)
print("Optimization complete!")

In [ ]:
# 6. Generate and save final comparison plot dashboard
plt.style.use('dark_background')
fig, axes = plt.subplots(2, 3, figsize=(15, 9), facecolor='#0b0f19')

for ax in axes.ravel():
    ax.set_facecolor('#0f172a')
    
# Row 1: Images
axes[0, 0].imshow(content_np)
axes[0, 0].set_title("Content Image", color='#4D96FF', fontsize=12, fontweight='bold', pad=10)
axes[0, 0].axis('off')

axes[0, 1].imshow(style_np)
axes[0, 1].set_title("Style Image", color='#FFB319', fontsize=12, fontweight='bold', pad=10)
axes[0, 1].axis('off')

axes[0, 2].imshow(output_np)
axes[0, 2].set_title("Stylised Output", color='#2ec4b6', fontsize=12, fontweight='bold', pad=10)
axes[0, 2].axis('off')

# Row 2: Plots
axes[1, 0].plot(steps_list, content_losses, color='#4D96FF', linewidth=2)
axes[1, 0].set_title("Content Loss", color='#f8fafc', fontsize=11, pad=8)
axes[1, 0].set_xlabel("Step", color='#94a3b8')
axes[1, 0].grid(True, color='#334155', linestyle='--', alpha=0.5)

axes[1, 1].plot(steps_list, style_losses, color='#FFB319', linewidth=2)
axes[1, 1].set_title("Style Loss", color='#f8fafc', fontsize=11, pad=8)
axes[1, 1].set_xlabel("Step", color='#94a3b8')
axes[1, 1].grid(True, color='#334155', linestyle='--', alpha=0.5)

axes[1, 2].plot(steps_list, total_losses_normalized, color='#2ec4b6', linewidth=2)
axes[1, 2].set_title("Total Loss (normalised)", color='#f8fafc', fontsize=11, pad=8)
axes[1, 2].set_xlabel("Step", color='#94a3b8')
axes[1, 2].grid(True, color='#334155', linestyle='--', alpha=0.5)

fig.suptitle("Neural Style Transfer -- Results", color='#f8fafc', fontsize=16, fontweight='bold', y=0.96)

plt.tight_layout(rect=[0, 0.03, 1, 0.93])
comparison_path = os.path.join(output_dir, 'comparison.png')
plt.savefig(comparison_path, facecolor=fig.get_facecolor(), edgecolor='none', dpi=150)
plt.show()